In [1]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q
!pip install databricks-sql-connector pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

Mounted at /content/drive
✅ Drive mounted successfully


In [3]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [4]:
import boto3
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql import DataFrame
from functools import reduce
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Download All Partitioned Files from MinIO ─────────────────────────────────
local_dir  = "/tmp/dividend"
bucket     = "rawdatasets"
prefix     = "dividend/"

os.makedirs(local_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    s3_key = obj["Key"]

    # Skip if the key is just the folder prefix itself
    if s3_key.endswith("/"):
        continue

    # Reconstruct local path preserving year=/month= structure
    relative_path = os.path.relpath(s3_key, prefix)
    local_file_path = os.path.join(local_dir, relative_path)

    # Create subdirectories if needed (e.g. year=2023/month=6/)
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_file_path)
    #print(f"Downloaded: {s3_key} → {local_file_path}")

# ── Read All Partitions into Spark DataFrame ──────────────────────────────────
spark = SparkSession.builder \
    .appName("ReaddividendCSV") \
    .getOrCreate()

dfs = []

schema = StructType([
    StructField("date",                StringType(), nullable=True),
    StructField("transaction_details", StringType(), nullable=True),
    StructField("currency",            StringType(), nullable=True),
    StructField("money_in",            StringType(), nullable=True)
])

for filename in os.listdir(local_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(local_dir, filename)

        # Get file metadata
        modified_time = os.path.getmtime(filepath)
        date_modified = datetime.fromtimestamp(modified_time).strftime("%Y-%m-%d")

        # Read each CSV and append metadata columns
        df = spark.read \
            .option("header", "true") \
            .schema(schema) \
            .csv(filepath)

        df = df.withColumn("source_file", F.lit(filename)) \
               .withColumn("date_modified", F.lit(date_modified))

        dfs.append(df)

# Union all DataFrames into one
final_df = reduce(DataFrame.union, dfs)

#final_df.printSchema()
#final_df.show()


In [5]:
# dataframe changes
from pyspark.sql.functions import udf
from pyspark.sql.types import DateType
from dateutil import parser

final_df = final_df.toDF("date", "transaction_details", "currency", "money_in","source_file","date_modified")
final_df = final_df.dropna(how='any', thresh=None, subset=["date"])

#df.printSchema()
#df.show(100, truncate=False)

In [6]:

#Check date valid
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
from dateutil import parser

def try_parse_date(date_str):
    if date_str is None:
        return "NULL"
    try:
        parsed = parser.parse(date_str.strip())
        return parsed.strftime("%Y-%m-%d")  # return standardized date if valid
    except:
        return f"INVALID: {date_str}"  # return original value flagged as invalid

audit_udf = udf(try_parse_date, StringType())

df_date = final_df.withColumn("date_audit", audit_udf("date"))
#df_date.show(truncate=False)

valid_df   = df_date.filter(~col("date_audit").startswith("INVALID") & (col("date_audit") != "NULL"))
invalid_df = df_date.filter((col("date_audit").startswith("INVALID") | (col("date_audit") == "NULL")) & (col("date_audit") != "INVALID: Date"))

print("=== VALID DATES ===")
# valid_df.select("date", "date_audit").show(truncate=False)
valid_df = valid_df.withColumn("money_in", col("money_in").cast("double"))
valid_df = valid_df.withColumn("date_audit", col("date_audit").cast("date"))
valid_df.show(truncate=False)
valid_df.printSchema()
valid_df.count()

print("=== INVALID DATES ===")
invalid_df.select("date", "date_audit").show(truncate=False)

=== VALID DATES ===
+-----------+------------------------------------------------------------------------------------------------+--------+--------+-----------+-------------+----------+
|date       |transaction_details                                                                             |currency|money_in|source_file|date_modified|date_audit|
+-----------+------------------------------------------------------------------------------------------------+--------+--------+-----------+-------------+----------+
|22-Jan-2026|IBG CREDIT|UOAREIT40|2001626022710226483||16082221/D/UOA REIT|DIVIDEND PAYMENT                  |MYR     |39.2    |2026.csv   |2026-03-08   |2026-01-22|
|22-Jan-2026|IBG CREDIT|UOAREIT40|2001626022710226480||16082221/D/UOA REIT|DIVIDEND PAYMENT                  |MYR     |196.0   |2026.csv   |2026-03-08   |2026-01-22|
|20-Jan-2026|AUTOPAY CR|EDIVPANAMY005467|347167392105|EDIVPANAMY005467|BOARDROOM/PANASONIC|Transfer From CIMB|MYR     |15.0    |2026.csv   |2026-03-08

In [7]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "dividend"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [ ]:
from databricks import sql

df_to_insert = valid_df

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

COLUMNS = [
    "date",
    "transaction_details",
    "currency",
    "money_in",
    "source_file",
    "date_modified",
    "date_audit"
]

placeholders = ", ".join(["?"] * len(COLUMNS))
col_names    = ", ".join(COLUMNS)

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
                date                STRING,
                transaction_details STRING,
                currency            STRING,
                money_in            DOUBLE,
                source_file         STRING,
                date_modified       DATE,
                date_audit          DATE
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────

        # Spark equivalent of: [tuple(r) for r in df.itertuples(index=False)]
        rows = [tuple(row) for row in df_to_insert.collect()]

        insert_sql = f"""
        INSERT INTO {FULL_TABLE} ({col_names})
        SELECT {placeholders}
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE}
            WHERE date = ?
              AND transaction_details = ?
              AND source_file = ?
        )
        """


        # Append the 3 key values at end of each row for the WHERE NOT EXISTS check
        rows_with_keys = [
            row + (row[COLUMNS.index("date")],
                  row[COLUMNS.index("transaction_details")],
                  row[COLUMNS.index("source_file")])
            for row in rows
        ]

        cursor.executemany(insert_sql, rows_with_keys)

        print(f"✅ Inserted {df.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.dividend' created/verified.
